# S11 - Motores de reglas (Pagarium)

**Objetivo (RA5-b):** pasar del micro-motor a los motores de producción: **DMN**, **GoRules ZEN** y `rule-engine`, y medir cuándo compensa.

> Una **tabla de decisión** es una lista de filas "si se cumplen estas condiciones → esta decisión".

### Python en un vistazo (lo justo para esta sesión)

| Lo que verás | Qué significa |
|---|---|
| `# texto` | **Comentario**: Python lo ignora; es una nota para ti |
| `nombre = valor` | **Guarda** un valor con un nombre |
| `{ "clave": "valor" }` | Un **diccionario**: parejas nombre → valor |
| `[a, b, c]` | Una **lista**: valores ordenados |
| `def nombre(...):` | Crea una **función** |
| `"texto"` / `'texto'` | Un **texto** (string), entre comillas |
| `==` / `<` / `>=` | Comparaciones: igual / menor / mayor o igual |
| `and` / `or` | Y lógico / O lógico |
| `if ...:` / `for ... in ...:` | Condición / repetición |
| `print(...)` | Escribe en pantalla |

Ejecuta las celdas de arriba a abajo; los comentarios `#` explican cada línea.

## 1. Política de Pagarium como tabla DMN

| Importe | Antigüedad (meses) | Intentos | Decisión |
|---|---|---|---|
| `< 500` | `>= 6` | `< 4` | Aprobar |
| `< 500` | `>= 6` | `>= 4` | Revisar |
| `>= 500` | — | — | Revisar |
| — | `< 6` | — | Rechazar |

El guion `—` significa "no importa". Hit policy propuesta: **unique** (una sola fila puede encajar).

## 2. `rule-engine` en Python puro

`rule_engine` permite escribir condiciones **como texto** y evaluarlas contra un diccionario de hechos.

In [ ]:
%pip install rule-engine
import rule_engine

# Un diccionario con los hechos del pago.
ctx = {"importe": 200, "n_intentos": 5, "antiguedad_meses": 10}

# Rule(...) guarda una condición escrita en texto legible.
aprueba = rule_engine.Rule("importe < 500 and antiguedad_meses >= 6 and n_intentos < 4")
rechaza = rule_engine.Rule("n_intentos >= 4 or antiguedad_meses < 6")

# .matches(ctx) devuelve True si la condición se cumple con esos hechos.
print("aprueba:", aprueba.matches(ctx))   # False: n_intentos=5 no es < 4
print("rechaza:", rechaza.matches(ctx))   # True: n_intentos=5 >= 4

### Otra tabla: descuentos por categoría

`rule-engine` con dos condiciones (categoría e importe). Fíjate en que los textos van entre comillas simples.

In [ ]:
reglas = {
    "descuento_20": rule_engine.Rule("categoria == 'vip' and importe >= 100"),
    "descuento_10": rule_engine.Rule("categoria == 'vip' and importe < 100"),
    "descuento_5":  rule_engine.Rule("categoria == 'normal' and importe >= 100"),
    "sin_descuento": rule_engine.Rule("categoria == 'normal' and importe < 100"),
}
def decidir(c):                       # devuelve qué reglas encajan con los hechos c
    return [n for n, r in reglas.items() if r.matches(c)]

for c in [{"categoria": "vip", "importe": 200},    # vip y 200 -> descuento_20
          {"categoria": "normal", "importe": 50}]:  # normal y 50 -> sin_descuento
    print(c, "->", decidir(c))

## 3. Actividad — añade la regla de "Revisar"

Implementa la tabla DMN completa con `rule-engine` (o con tu `MicroMotor2` de la S9) y pruébala con 4 casos que cubran las 4 filas. Añade un 5º caso para detectar solapes.

In [ ]:
# TODO: implementa la politica completa (4 reglas) y prueba 4 casos
...

## 4. GoRules ZEN (fintech)

ZEN ejecuta un grafo de decisión (formato **JDM**, JSON). Normalmente se dibuja en su editor web y se exporta. La API cambió: los inputs/outputs llevan `"ref": ""` y los valores de salida van como texto JSON.

In [ ]:
%pip install zen-engine
import json
from zen import ZenEngine

# Un grafo de decisión: nodo de entrada -> tabla -> nodo de salida.
graph = {
    "contentType": "application/vnd.gorules.decision",
    "nodes": [
        {"id": "input", "type": "inputNode", "name": "Request", "position": {"x": 0, "y": 0}},
        {"id": "table", "type": "decisionTableNode", "name": "Politica",
         "content": {
             "hitPolicy": "first",     # gana la primera fila que encaja
             "inputs":  [{"id": "i1", "field": "importe", "name": "Importe", "ref": ""}],
             "outputs": [{"id": "o1", "field": "decision", "name": "Decision", "ref": ""}],
             "rules": [
                 {"i1": "< 500",  "_id": "r1", "o1": "\"aprobar\""},   # el valor es un texto JSON
                 {"i1": ">= 500", "_id": "r2", "o1": "\"revisar\""},
             ],
         },
         "position": {"x": 200, "y": 0}},
        {"id": "out", "type": "outputNode", "name": "Response",
         "content": {"fields": [{"id": "decision", "field": "decision"}]},
         "position": {"x": 400, "y": 0}},
    ],
    "edges": [
        {"id": "e1", "sourceId": "input", "targetId": "table"},
        {"id": "e2", "sourceId": "table", "targetId": "out"},
    ],
}
engine = ZenEngine()
decision = engine.create_decision(json.dumps(graph))   # json.dumps convierte el dict a texto JSON
print(decision.evaluate(json.dumps({"importe": 200})))  # evalúa con importe 200

### Grafo ZEN con dos entradas

La tabla ahora decide por importe **y** número de intentos.

In [ ]:
graph2 = {
  "contentType": "application/vnd.gorules.decision",
  "nodes": [
    {"id": "input", "type": "inputNode", "name": "Request", "position": {"x": 0, "y": 0}},
    {"id": "table", "type": "decisionTableNode", "name": "Politica",
     "content": {
       "hitPolicy": "first",
       "inputs": [
         {"id": "i1", "field": "importe", "name": "Importe", "ref": ""},
         {"id": "i2", "field": "n_intentos", "name": "Intentos", "ref": ""},
       ],
       "outputs": [{"id": "o1", "field": "decision", "name": "Decision", "ref": ""}],
       "rules": [
         {"_id": "r1", "i1": "< 500", "i2": "< 4", "o1": "\"aprobar\""},
         {"_id": "r2", "i1": "< 500", "i2": ">= 4", "o1": "\"revisar\""},
         {"_id": "r3", "i1": ">= 500", "i2": "-", "o1": "\"revisar\""},   # "-" = no importa
       ],
     },
     "position": {"x": 200, "y": 0}},
    {"id": "out", "type": "outputNode", "name": "Response",
     "content": {"fields": [{"id": "decision", "field": "decision"}]},
     "position": {"x": 400, "y": 0}},
  ],
  "edges": [
    {"id": "e1", "sourceId": "input", "targetId": "table"},
    {"id": "e2", "sourceId": "table", "targetId": "out"},
  ],
}
decision2 = engine.create_decision(json.dumps(graph2))
print(decision2.evaluate(json.dumps({"importe": 200, "n_intentos": 2})))   # -> aprobar
print(decision2.evaluate(json.dumps({"importe": 200, "n_intentos": 5})))   # -> revisar

## 5. Benchmark: ¿compensa un motor?

Un motor de reglas es más cómodo pero **mucho más lento** que un `if/else`. Medimos cuánto.

In [ ]:
import timeit   # timeit mide el tiempo de ejecutar algo muchas veces

ctx = {"importe": 200, "n_intentos": 5, "antiguedad_meses": 10}

def ifelse(c):                        # la misma decisión, escrita a mano con if/else
    if c["importe"] < 500 and c["antiguedad_meses"] >= 6 and c["n_intentos"] < 4:
        return "aprobar"
    if c["n_intentos"] >= 4 or c["antiguedad_meses"] < 6:
        return "rechazar"
    return "revisar"

# timeit.timeit(ejecuta la función 100000 veces) / 100000 = tiempo medio; * 1e6 = microsegundos
print("if/else:", round(timeit.timeit(lambda: ifelse(ctx), number=100000) / 100000 * 1e6, 3), "us")

**Actividad:** mide con `timeit` tu `if/else` frente a `rule-engine` para el mismo caso y calcula el factor (cuántas veces es más lento el motor).